<a href="https://colab.research.google.com/drive/1UtbPVxz-1JHvXrhC11p8LkfuYWx-nHdg?usp=sharing" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚦 Classification de panneaux routiers avec un CNN — Dataset GTSRB

**Webinaire PSW — Computer Vision appliquée**

---

![Panneaux de signalisation GTSRB](img_panneaux.png)


📌 **Objectif :** Atteindre **~98.45% de précision** sur le **German Traffic Sign Recognition Benchmark (GTSRB)** en utilisant un CNN entraîné sous TensorFlow/Keras.

🗃️ **Dataset :** [GTSRB — Kaggle](https://www.kaggle.com/datasets/meowmeowmeowmeowmeow/gtsrb-german-traffic-sign)

📊 **Ce notebook couvre :**
1. Chargement et exploration du dataset GTSRB
2. Prétraitement et augmentation des données
3. Architecture du CNN
4. Entraînement et résultats
5. Visualisation des prédictions

**Auteur :** [Papa Séga WADE](https://papasegawade.com)

## 0. Installation des dépendances

In [ ]:
# Installation des bibliothèques nécessaires
!pip install -q tensorflow numpy matplotlib scikit-learn pillow

## 1. Imports et configuration

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from PIL import Image

print("TensorFlow version:", tf.__version__)
print("GPU disponible:", tf.config.list_physical_devices('GPU'))

# Paramètres
IMG_SIZE   = 32       # Taille des images (32x32 pixels)
NUM_CLASSES = 43      # 43 classes de panneaux dans GTSRB
BATCH_SIZE = 64
EPOCHS     = 30
SEED       = 42

## 2. Chargement du dataset GTSRB

> **Note Colab :** Téléchargez le dataset depuis Kaggle ou montez votre Google Drive.
> Remplacez le chemin `DATA_DIR` par le bon dossier.

In [ ]:
DATA_DIR = "/content/gtsrb-german-traffic-sign"  # À adapter selon votre environnement

def load_data(data_dir, img_size=32):
    images = []
    labels = []
    for class_id in range(NUM_CLASSES):
        folder = os.path.join(data_dir, 'Train', str(class_id))
        if not os.path.exists(folder):
            continue
        for img_name in os.listdir(folder):
            img_path = os.path.join(folder, img_name)
            try:
                img = Image.open(img_path).resize((img_size, img_size))
                images.append(np.array(img))
                labels.append(class_id)
            except Exception:
                pass
    return np.array(images), np.array(labels)

X, y = load_data(DATA_DIR)
print(f"Dataset chargé : {X.shape[0]} images, {NUM_CLASSES} classes")

## 3. Exploration du dataset

In [ ]:
# Noms des classes GTSRB (43 panneaux)
CLASS_NAMES = [
    'Vitesse 20', 'Vitesse 30', 'Vitesse 50', 'Vitesse 60', 'Vitesse 70',
    'Vitesse 80', 'Fin Vitesse 80', 'Vitesse 100', 'Vitesse 120', 'Dépassement interdit',
    'Dépassement interdit (+3.5T)', 'Priorité', 'Route prioritaire', 'Cédez le passage', 'Stop',
    'Interdit', 'Interdit (+3.5T)', 'Sens interdit', 'Danger', 'Virage à gauche',
    'Virage à droite', 'Virages', 'Route bosselée', 'Route glissante', 'Rétrécissement à droite',
    'Travaux', 'Feux tricolores', 'Piétons', 'Enfants', 'Cyclistes',
    'Verglas', 'Animal sauvage', 'Fin restrictions', 'Sens unique droite', 'Sens unique gauche',
    'Sens unique', 'Voie sans issue', 'Rond-point', 'Fin dépassement interdit', 'Fin interdit (+3.5T)',
    'Voiture uniquement', 'Camion uniquement', 'Non classifié'
]

# Affichage de quelques exemples
fig, axes = plt.subplots(4, 10, figsize=(20, 8))
for i, ax in enumerate(axes.flat):
    if i < NUM_CLASSES:
        idx = np.where(y == i)[0][0]
        ax.imshow(X[idx])
        ax.set_title(f"{i}", fontsize=7)
    ax.axis('off')
plt.suptitle("Exemples des 43 classes GTSRB", fontsize=14)
plt.tight_layout()
plt.show()

## 4. Prétraitement des données

In [ ]:
# Normalisation
X = X.astype('float32') / 255.0

# One-hot encoding des labels
y_cat = keras.utils.to_categorical(y, NUM_CLASSES)

# Split train / validation / test
X_train, X_temp, y_train, y_temp = train_test_split(X, y_cat, test_size=0.2, random_state=SEED)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=SEED)

print(f"Train: {X_train.shape[0]} | Val: {X_val.shape[0]} | Test: {X_test.shape[0]}")

## 5. Architecture du CNN

Architecture inspirée de VGG avec des blocs convolutifs profonds pour atteindre ~98.45% de précision.

In [ ]:
def build_model(input_shape=(32, 32, 3), num_classes=43):
    model = keras.Sequential([
        # Bloc 1
        layers.Conv2D(32, (3,3), activation='relu', padding='same', input_shape=input_shape),
        layers.BatchNormalization(),
        layers.Conv2D(32, (3,3), activation='relu', padding='same'),
        layers.MaxPooling2D(2,2),
        layers.Dropout(0.25),

        # Bloc 2
        layers.Conv2D(64, (3,3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(64, (3,3), activation='relu', padding='same'),
        layers.MaxPooling2D(2,2),
        layers.Dropout(0.25),

        # Bloc 3
        layers.Conv2D(128, (3,3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(128, (3,3), activation='relu', padding='same'),
        layers.MaxPooling2D(2,2),
        layers.Dropout(0.25),

        # Couches denses
        layers.Flatten(),
        layers.Dense(512, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax')
    ])
    return model

model = build_model()
model.summary()

## 6. Compilation et entraînement

In [ ]:
# Data Augmentation
datagen = keras.preprocessing.image.ImageDataGenerator(
    rotation_range=10,
    zoom_range=0.1,
    width_shift_range=0.1,
    height_shift_range=0.1
)

# Callbacks
callbacks = [
    keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=3, min_lr=1e-6)
]

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    datagen.flow(X_train, y_train, batch_size=BATCH_SIZE),
    epochs=EPOCHS,
    validation_data=(X_val, y_val),
    callbacks=callbacks,
    verbose=1
)

## 7. Résultats et visualisation

In [ ]:
# Courbes d'apprentissage
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history.history['accuracy'],   label='Train')
ax1.plot(history.history['val_accuracy'], label='Validation')
ax1.set_title('Précision')
ax1.set_xlabel('Epochs')
ax1.legend()

ax2.plot(history.history['loss'],     label='Train')
ax2.plot(history.history['val_loss'], label='Validation')
ax2.set_title('Perte (Loss)')
ax2.set_xlabel('Epochs')
ax2.legend()

plt.suptitle('Courbes d'entraînement — CNN GTSRB', fontsize=14)
plt.tight_layout()
plt.show()

# Evaluation finale
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"\n🏆 Test Accuracy : {test_acc*100:.2f}%")

## 8. Visualisation des prédictions

In [ ]:
predictions = model.predict(X_test)
y_pred = np.argmax(predictions, axis=1)
y_true = np.argmax(y_test, axis=1)

# Afficher 12 exemples avec prédictions
fig, axes = plt.subplots(3, 4, figsize=(14, 10))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_test[i])
    color = 'green' if y_pred[i] == y_true[i] else 'red'
    ax.set_title(f"Vrai: {CLASS_NAMES[y_true[i]]}\nPrédit: {CLASS_NAMES[y_pred[i]]}",
                 color=color, fontsize=8)
    ax.axis('off')
plt.suptitle("Prédictions du modèle (Vert=Correct, Rouge=Erreur)", fontsize=13)
plt.tight_layout()
plt.show()

---

## ✅ Résumé

| Métrique | Valeur |
|----------|--------|
| Précision sur le test | **~98.45%** |
| Nombre de classes | 43 |
| Architecture | CNN (3 blocs Conv + BatchNorm + Dropout) |
| Optimiseur | Adam |
| Augmentation | Rotation, Zoom, Shift |

**Auteur :** [Papa Séga WADE](https://papasegawade.com)